In [1]:
%pip install transformers datasets torch jsonlines

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import BertTokenizer
import torch

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Preprocessing for MLM
def preprocess_for_mlm(context_list, tokenizer, max_length=512):
    """
    Prepares context data for MLM pretraining.
    - context_list: Nested list of paragraphs or sentences from the 'context' field.
    - tokenizer: BERT tokenizer.
    - max_length: Maximum sequence length.
    """
    def flatten(nested_list):
        """Recursively flattens a nested list."""
        for item in nested_list:
            if isinstance(item, list):
                yield from flatten(item)  # Recursively flatten
            else:
                yield item

    # Flatten the context and filter out non-string elements
    flat_context = [text for text in flatten(context_list) if isinstance(text, str)]

    # Join all strings into a single text
    text = " ".join(flat_context)

    # Tokenize the text
    tokens = tokenizer.tokenize(text)

    # Truncate to max_length (leave space for [CLS] and [SEP])
    if len(tokens) > max_length - 2:
        tokens = tokens[:max_length - 2]

    # Add [CLS] and [SEP]
    tokens = ["[CLS]"] + tokens + ["[SEP]"]

    # Convert tokens to IDs
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    # Create MLM labels (-100 for non-masked tokens)
    labels = [-100] * len(token_ids)
    num_to_mask = int(0.15 * len(token_ids))  # Mask 15%

    # Mask tokens (example random selection)
    for i in torch.randperm(len(token_ids))[:num_to_mask]:
        if token_ids[i] not in [tokenizer.cls_token_id, tokenizer.sep_token_id]:
            labels[i] = token_ids[i]
            token_ids[i] = tokenizer.mask_token_id

    return token_ids, labels

# Example usage
with open("hotpot_train_v1.1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

examples = []
for item in data:
    token_ids, labels = preprocess_for_mlm(item["context"], tokenizer)
    examples.append((token_ids, labels))


In [10]:
# Load the dataset
import json
from transformers import BertTokenizer

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Load JSON file
with open("hotpot_train_v1.1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Process the data
examples = []
for item in data:
    token_ids, labels = preprocess_for_mlm(item["context"], tokenizer)
    examples.append((token_ids, labels))

# Save the processed data
import torch
torch.save(examples, "mlm_training_data.pt")
print("Preprocessed data saved successfully!")


Preprocessed data saved successfully!


In [ ]:
import torch
from transformers import BertForMaskedLM, BertTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import json
import random
from sklearn.model_selection import train_test_split

# Step 1: Load and preprocess the dataset
class MLM_Dataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        # This function is responsible for tokenizing the data
        token_ids, labels = self.examples[idx]
        
        # Pad the sequences to max_length
        padding_length = self.max_length - len(token_ids)
        if padding_length > 0:
            token_ids = token_ids + [self.tokenizer.pad_token_id] * padding_length
            labels = labels + [-100] * padding_length  # Masked tokens don't contribute to loss

        return {
            "input_ids": torch.tensor(token_ids),
            "labels": torch.tensor(labels)
        }

def preprocess_for_mlm(context_list, tokenizer, max_length=512):
    """
    Prepares context data for MLM pretraining.
    - context_list: List of paragraphs or sentences from the 'context' field.
    - tokenizer: BERT tokenizer.
    - max_length: Maximum sequence length.
    """
    # Flatten and join the context into a single string
    flat_context = []
    for paragraph in context_list:
        if isinstance(paragraph, list):  # Each paragraph could have multiple sentences
            for sentence in paragraph:  # Ensure each sentence is a string
                if isinstance(sentence, str):  # Add the sentence if it is a string
                    flat_context.append(sentence)
        elif isinstance(paragraph, str):  # If it's a string, append it
            flat_context.append(paragraph)

    # Join all strings into a single text
    text = " ".join(flat_context)

    # Tokenize the text
    tokens = tokenizer.tokenize(text)

    # Make sure the length is within the max limit (accounting for special tokens)
    if len(tokens) > max_length - 2:  # Reserve space for [CLS] and [SEP]
        tokens = tokens[:max_length - 2]

    # Add special tokens ([CLS] and [SEP])
    tokens = ["[CLS]"] + tokens + ["[SEP]"]
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    # Create masked labels (for MLM) by randomly masking some of the tokens
    labels = token_ids.copy()
    mask_probability = 0.15  # Standard practice for masking
    for i in range(1, len(token_ids) - 1):  # Skip the [CLS] and [SEP] tokens
        if random.random() < mask_probability:
            labels[i] = tokenizer.convert_tokens_to_ids("[MASK]")

    return token_ids, labels

# Load the pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# Load dataset
with open("hotpot_train_v1.1.json", "r") as f:
    data = json.load(f)

# Preprocess the data
examples = []
for item in data:
    context = item["context"]
    token_ids, labels = preprocess_for_mlm(context, tokenizer)
    examples.append((token_ids, labels))

# Split the dataset into train and eval (validation) sets
train_examples, eval_examples = train_test_split(examples, test_size=0.1)  # 10% for validation

# Create Dataset and DataLoader for train and eval
train_dataset = MLM_Dataset(train_examples, tokenizer)
eval_dataset = MLM_Dataset(eval_examples, tokenizer)

# Step 2: Training Arguments
training_args = TrainingArguments(
    output_dir="./mlm_model",          # Save the model here
    overwrite_output_dir=True,         # Overwrite output directory if it exists
    num_train_epochs=3,                # Number of epochs
    per_device_train_batch_size=8,     # Batch size
    save_steps=1000,                   # Save checkpoint every 1000 steps
    logging_dir="./logs",              # Directory for logs
    logging_steps=500,                 # Log every 500 steps
    evaluation_strategy="steps",       # Evaluate during training
    eval_steps=500,                    # Evaluate every 500 steps
    save_total_limit=3,                # Keep only the latest 3 checkpoints
    do_train=True,                     # Enable training
    do_eval=True,                      # Enable evaluation
)

# Step 3: Trainer
trainer = Trainer(
    model=model,                      # The model to train
    args=training_args,               # The training arguments
    train_dataset=train_dataset,      # Your training dataset
    eval_dataset=eval_dataset,        # Your evaluation dataset
)

# Step 4: Training
# trainer.train()
trainer.train(resume_from_checkpoint="./mlm_model/checkpoint-7000")


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].
C:\Users\kavin\AppData\Roaming\Python\Python312\site-packages\transformers\trainer.py:3420: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), 

  0%|          | 0/30528 [00:00<?, ?it/s]

C:\Users\kavin\AppData\Roaming\Python\Python312\site-packages\transformers\trainer.py:3083: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_rng_state = torch.load(r

{'loss': 0.4619, 'grad_norm': 1.623064637184143, 'learning_rate': 4.4267557651991616e-05, 'epoch': 0.34}


  0%|          | 0/1131 [00:00<?, ?it/s]

...

  0%|          | 0/1131 [00:00<?, ?it/s]

{'eval_loss': 0.5027814507484436, 'eval_runtime': 3261.7537, 'eval_samples_per_second': 2.773, 'eval_steps_per_second': 0.347, 'epoch': 0.64}
{'loss': 0.4495, 'grad_norm': 1.349916934967041, 'learning_rate': 3.8535115303983236e-05, 'epoch': 0.69}


  0%|          | 0/1131 [00:00<?, ?it/s]

{'eval_loss': 0.4828279912471771, 'eval_runtime': 3218.6499, 'eval_samples_per_second': 2.81, 'eval_steps_per_second': 0.351, 'epoch': 0.69}
{'loss': 0.446, 'grad_norm': 1.5632658004760742, 'learning_rate': 3.771619496855346e-05, 'epoch': 0.74}


  0%|          | 0/1131 [00:00<?, ?it/s]